In [1]:
# To support both python 2 and python 3
from __future__ import division, print_function, unicode_literals

# Common imports
import numpy as np
import pandas as pd
import os

# Plot settings
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# XGBoost
from xgboost import XGBRegressor

RANDOM_STATE = 42

In [2]:
# Load data
df = pd.read_csv("cleaned_car_data.csv")

# Basic checks
print(df.shape)
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

(94435, 10)
  model  year  price transmission fueltype    tax  enginesize brand  \
0    A1  2017  12500       Manual   Petrol  150.0         1.4  audi   
1    A6  2016  16500    Automatic   Diesel   20.0         2.0  audi   
2    A1  2016  11000       Manual   Petrol   30.0         1.4  audi   
3    A4  2017  16800    Automatic   Diesel  145.0         2.0  audi   
4    A3  2019  17300       Manual   Petrol  145.0         1.0  audi   

   km_per_litre  mileage_km  
0            19       25323  
1            22       58263  
2            19       48193  
3            23       41765  
4            17        3215  

Data types:
model            object
year              int64
price             int64
transmission     object
fueltype         object
tax             float64
enginesize      float64
brand            object
km_per_litre      int64
mileage_km        int64
dtype: object

Missing values:
model           0
year            0
price           0
transmission    0
fueltype        0
tax    

In [3]:
# Separate features and target
X = df.drop("price", axis=1)
y = df["price"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (94435, 9)
y shape: (94435,)


In [4]:
# Numerical and categorical columns
num_attribs = X.select_dtypes(include=[np.number]).columns.tolist()
cat_attribs = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numerical attributes:", num_attribs)
print("Categorical attributes:", cat_attribs)

Numerical attributes: ['year', 'tax', 'enginesize', 'km_per_litre', 'mileage_km']
Categorical attributes: ['model', 'transmission', 'fueltype', 'brand']


In [5]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (75548, 9)
X_test: (18887, 9)
y_train: (75548,)
y_test: (18887,)


In [6]:
# Numerical pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical pipeline
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Full preprocessing
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs)
])

In [7]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name="Model"):
    # Train
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Metrics
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    print("=" * 50)
    print(model_name)
    print("=" * 50)
    print("Train MAE:  ", round(train_mae, 3))
    print("Test MAE:   ", round(test_mae, 3))
    print("Train RMSE: ", round(train_rmse, 3))
    print("Test RMSE:  ", round(test_rmse, 3))
    print("Train R2:   ", round(train_r2, 4))
    print("Test R2:    ", round(test_r2, 4))
    
    return {
        "Model": model_name,
        "Train_MAE": train_mae,
        "Test_MAE": test_mae,
        "Train_RMSE": train_rmse,
        "Test_RMSE": test_rmse,
        "Train_R2": train_r2,
        "Test_R2": test_r2
    }

In [8]:
# Linear Regression pipeline
lin_reg_pipeline = Pipeline([
    ("preprocessing", full_pipeline),
    ("model", LinearRegression())
])

lin_reg_results = evaluate_model(
    lin_reg_pipeline,
    X_train, y_train,
    X_test, y_test,
    model_name="Linear Regression"
)

Linear Regression
Train MAE:   2434.793
Test MAE:    2416.974
Train RMSE:  3389.176
Test RMSE:   3358.552
Train R2:    0.8333
Test R2:     0.8342


In [9]:
# Random Forest pipeline
rf_pipeline = Pipeline([
    ("preprocessing", full_pipeline),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_results = evaluate_model(
    rf_pipeline,
    X_train, y_train,
    X_test, y_test,
    model_name="Random Forest Regressor"
)

Random Forest Regressor
Train MAE:   671.337
Test MAE:    1111.356
Train RMSE:  1016.671
Test RMSE:   1680.255
Train R2:    0.985
Test R2:     0.9585


In [10]:
# XGBoost pipeline
xgb_pipeline = Pipeline([
    ("preprocessing", full_pipeline),
    ("model", XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

xgb_results = evaluate_model(
    xgb_pipeline,
    X_train, y_train,
    X_test, y_test,
    model_name="XGBoost Regressor"
)

XGBoost Regressor
Train MAE:   1196.608
Test MAE:    1234.616
Train RMSE:  1690.588
Test RMSE:   1773.517
Train R2:    0.9585
Test R2:     0.9538


In [11]:
results_df = pd.DataFrame([
    lin_reg_results,
    rf_results,
    xgb_results
])

results_df = results_df.sort_values("Test_RMSE")
results_df

,Model,Train_MAE,Test_MAE,Train_RMSE,Test_RMSE,Train_R2,Test_R2
1,Random Forest Regressor,671.337097,1111.355959,1016.670908,1680.254806,0.984995,0.958497
2,XGBoost Regressor,1196.607910,1234.615967,1690.587768,1773.516845,0.958510,0.953762
0,Linear Regression,2434.792865,2416.974460,3389.175951,3358.551697,0.833255,0.834181
